# NB3 · Ghép, kiểm tra & xuất file nộp BTC
**FinReport Intelligence · Festival Chuyển đổi số 2026 · Bảng C · Case 3**

| | |
|---|---|
| **Đọc từ** | `03_outputs/step1_3/` (chỉ tiêu, câu trả lời QA) · `03_outputs/step4_6/predictions.csv` (cảnh báo) · template BTC |
| **Ghi ra** | `05_submission/submission_output.csv` · `05_submission/qa_submission.csv` · `05_submission/submission_manifest.csv` · `03_outputs/step7_submit/` |
| **Thư viện** | `lib/finreport_submit.py` (+ common, metrics, signals, qa) |

Các bước: (1) nhận diện file Data Pack dù BTC đặt tên/thư mục khác README → (2) ghép đúng thứ tự cột & dòng template →
(3) **kiểm tra trước khi ghi** (schema, khóa, miền giá trị, quy ước 2021, nhất quán nội bộ) → (4) **đọc lại chính file sẽ nộp và chấm lại** →
(5) kiểm thử âm cho bộ kiểm tra → (6) ghi file + manifest SHA-256 + audit log.

## 0. Kết nối Drive, nạp thư viện, kiểm tra kết quả NB1 & NB2

In [ ]:
import os, sys, importlib, time, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/FinReport_Intelligence"
except ImportError:
    PROJECT = os.environ.get("FINREPORT_PROJECT", "FinReport_Intelligence")

sys.path.insert(0, PROJECT + "/lib")
import finreport_common, finreport_metrics, finreport_signals, finreport_qa, finreport_submit
for mod in (finreport_common, finreport_metrics, finreport_signals, finreport_qa, finreport_submit):
    importlib.reload(mod)
from finreport_common import *
import finreport_metrics as fm, finreport_signals as fs, finreport_qa as qa, finreport_submit as sb
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)

cfg = setup(PROJECT)
STEP = "step7_submit"
RUN_S7 = new_run_id("S7")

audit = read_audit()
last = {s: audit[audit["step"] == s].tail(1) for s in ["step3_verify_qa", "step6_scoring"]}
for s, r in last.items():
    assert len(r), f"Chưa có bản ghi {s} – hãy chạy NB1 và NB2 trước"
    print(f"{s:16s}: {r.iloc[0]['run_id']} · {r.iloc[0]['verdict']}")
PARENTS = [last["step3_verify_qa"].iloc[0]["run_id"], last["step6_scoring"].iloc[0]["run_id"]]

## 1. Nhận diện file Data Pack

BTC thực tế phát `03_sample/` thay vì `03_labels_sample/`, template có hậu tố `_Case3` ở thư mục gốc.
`locate_pack_file()` tìm theo thứ tự: **đường dẫn chuẩn README → cùng tên ở thư mục con → tên có hậu tố**, và ghi lại cách tìm thấy vào audit log.
Nhờ vậy nếu bộ dữ liệu chấm cuối đặt tên khác, hệ thống vẫn chạy và có vết để kiểm tra.

In [ ]:
layout = []
for name in {**CORE_FILES, **METADATA_FILES, **LABEL_FILES, **TEMPLATE_FILES}:
    p, how = locate_pack_file(name)
    layout.append({"file": name, "found": p is not None, "how": how,
                   "path": "" if p is None else str(p.relative_to(cfg.data_dir))})
layout = pd.DataFrame(layout)
display(layout)
missing = layout.loc[~layout["found"] & layout["file"].isin(list(CORE_FILES) + list(TEMPLATE_FILES)), "file"].tolist()
assert not missing, f"Thiếu file bắt buộc để xuất bài: {missing}"

## 2. Nạp kết quả các bước trước & template

In [ ]:
master = read_output("step0", "master_panel.csv")
metrics_wide = read_output("step1_3", "metrics_wide.csv")
supp = read_output("step1_3", "metrics_supplementary.csv")
qa_detail = read_output("step1_3", "qa_answers_detail.csv")
predictions = read_output("step4_6", "predictions.csv")
tpl_sub = read_pack("submission_output_template")
tpl_qa = read_pack("qa_submission_template")
for n, d_ in [("master_panel", master), ("metrics_wide", metrics_wide), ("predictions", predictions),
              ("qa_answers_detail", qa_detail), ("template submission", tpl_sub), ("template QA", tpl_qa)]:
    print(f"{n:22s} {d_.shape}")

## 3. Thời gian xử lý (`processing_time_ms`)

- **File cảnh báo**: chạy lại toàn bộ chuỗi *tính chỉ tiêu → đặc trưng → tín hiệu → chấm điểm → giải thích* 3 lần, lấy trung vị, **chia đều cho 60 dòng**
  (không tách riêng từng dòng được vì tín hiệu dùng benchmark của cả ngành).
- **File QA**: thời gian trả lời đo riêng từng câu ở NB1.

In [ ]:
def full_pipeline():
    w, _ = fm.compute_metrics(master)
    s_ = fm.compute_supplementary(master)
    return fs.run(master, w, s_)

ms_per_row, ms_total = sb.timing_per_row(master, full_pipeline, repeats=3)
print(f"Toàn pipeline: {ms_total:.1f} ms cho {len(master)} company-year → {ms_total/len(master):.2f} ms/dòng")
print(f"QA: trung bình {qa_detail['processing_time_ms'].mean():.3f} ms/câu")

## 4. Ghép file `submission_output` (60 dòng × 31 cột)

In [ ]:
submission = sb.assemble_submission(tpl_sub, metrics_wide, predictions, fs.MODEL_NAME, ms_per_row)
print(submission.shape)
display(submission.head(3).T)

In [ ]:
# Xem phần cảnh báo của một vài dòng
display(submission.loc[submission["priority_rank"] <= 5,
        ["company_id", "fiscal_year", "priority_rank", "risk_score", "predicted_severity", "predicted_overall_health",
         "predicted_primary_scenario", "alert_tags", "confidence"]].sort_values("priority_rank"))

## 5. Ghép file `qa_submission` (30 câu × 8 cột)

In [ ]:
qa_sub = sb.assemble_qa(tpl_qa, qa_detail)
print(qa_sub.shape)
display(qa_sub.head(8))

## 6. Kiểm tra trước khi ghi file

| Nhóm | Kiểm tra |
|---|---|
| V1 – submission | Đúng tên & thứ tự 31 cột · đúng 60 khóa theo thứ tự template · không trùng khóa · 2021 để trống 6 chỉ tiêu cần năm trước · các ô còn lại đủ giá trị · khớp bảng chỉ tiêu NB1 · `risk_score`, `confidence` ∈ [0,1] · severity/health đúng tập giá trị BTC · `priority_rank` nguyên dương, duy nhất · lý do, bằng chứng, model_name không rỗng · HEALTHY ⇒ LOW · CRITICAL ⇒ HIGH_RISK · bằng chứng đúng định dạng `file.csv:Cxxx:năm:trường` |
| V2 – QA | Đúng 8 cột · đúng 30 question_id theo thứ tự · đáp án, nguồn, giải thích, model_name không rỗng · confidence ∈ [0,1] · nguồn là file .csv |

In [ ]:
vlog = CheckLog()
sb.validate_submission(submission, tpl_sub, metrics_wide, vlog)
sb.validate_qa(qa_sub, tpl_qa, vlog)
VERDICT = vlog.verdict()
print("Kết luận kiểm tra:", VERDICT, vlog.counts())
display(vlog.df())
assert not vlog.failed, "Có lỗi FAIL – không ghi file nộp. Xem bảng trên."

## 7. Ghi file nộp & đọc lại để chấm

Ghi vào `05_submission/`, rồi **đọc lại chính file vừa ghi từ ổ đĩa** (không dùng biến trong bộ nhớ) để chấm:
- **QA**: tách đáp án từ chữ trong cột `answer` (số, mã DN, Có/Không, tag) → so answer key.
- **Chỉ tiêu**: so với `financial_metrics_sample.csv`.
- **Cảnh báo**: so với 18 nhãn mẫu.

Nếu file bị lỗi định dạng khi ghi (dấu phẩy, mã hóa, làm tròn…), bước này sẽ phát hiện.

In [ ]:
SUB_PATH = cfg.submit_dir / "submission_output.csv"
QA_PATH = cfg.submit_dir / "qa_submission.csv"
submission.to_csv(SUB_PATH, index=False, encoding="utf-8-sig")
qa_sub.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print("Đã ghi:", SUB_PATH.name, "·", QA_PATH.name)

sub_disk = pd.read_csv(SUB_PATH, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
qa_disk = pd.read_csv(QA_PATH, encoding="utf-8-sig")

final = {}
ak_path, _ = locate_pack_file("answer_key_sample")
if ak_path is not None:
    g = sb.regrade_qa_file(qa_disk, pd.read_csv(ak_path, encoding="utf-8-sig"))
    final["numerical_qa_accuracy_file"] = g["correct"].mean()
    print(f"QA chấm lại từ file nộp: {g['correct'].sum()}/{len(g)} = {g['correct'].mean():.1%}")
    if (~g["correct"]).any():
        display(g[~g["correct"]])

ms_path, _ = locate_pack_file("financial_metrics_sample")
if ms_path is not None:
    sub_m = sub_disk[fm.KEY + fm.METRIC_IDS].copy()
    cmp_ = fm.compare_with_metrics_sample(sub_m, supp, ms_path)
    tpl_c = cmp_[cmp_["in_template"]]
    final["calculation_accuracy_file"] = tpl_c["within_btc_tolerance"].mean()
    final["calculation_exact_file"] = tpl_c["exact_at_btc_precision"].mean()
    print(f"Chỉ tiêu trong file nộp vs metrics mẫu: trong dung sai {tpl_c['within_btc_tolerance'].sum()}/{len(tpl_c)} = "
          f"{final['calculation_accuracy_file']:.1%} · khớp tuyệt đối {final['calculation_exact_file']:.1%}")

lb_path, _ = locate_pack_file("financial_scenarios_sample")
if lb_path is not None:
    labels = pd.read_csv(lb_path, encoding="utf-8-sig", keep_default_na=False)
    ev = fs.evaluate(sub_disk.assign(alert_tags=sub_disk["alert_tags"].fillna("")), labels)
    final.update({f"alert_{k}_file": v for k, v in ev.items() if k in
                  ("tag_f1_micro", "scenario_accuracy", "severity_accuracy", "health_accuracy", "high_critical_recall")})
    print(f"Cảnh báo trong file nộp vs 18 nhãn mẫu: tag F1 {ev['tag_f1_micro']:.1%} · kịch bản {ev['scenario_accuracy']:.1%} · "
          f"severity {ev['severity_accuracy']:.1%} · health {ev['health_accuracy']:.1%}")

## 8. Kiểm thử âm – bộ kiểm tra có bắt được file lỗi không?

Cố ý làm hỏng **bản sao** file nộp theo 6 kiểu lỗi thường gặp, chạy lại bộ kiểm tra. Mong đợi: **mọi kiểu lỗi đều bị FAIL**.

In [ ]:
def corrupt(kind):
    s = submission.copy()
    if kind == "đổi tên cột":            s = s.rename(columns={"risk_score": "risk"})
    if kind == "thiếu 1 dòng":           s = s.iloc[:-1]
    if kind == "risk_score > 1":         s.loc[0, "risk_score"] = 1.7
    if kind == "severity sai tên":       s.loc[0, "predicted_severity"] = "SEVERE"
    if kind == "tự sinh số 2021":        s.loc[s["fiscal_year"] == 2021, "dso_days"] = 45.0
    if kind == "trùng priority_rank":    s.loc[1, "priority_rank"] = s.loc[0, "priority_rank"]
    return s

neg = []
for kind in ["đổi tên cột", "thiếu 1 dòng", "risk_score > 1", "severity sai tên", "tự sinh số 2021", "trùng priority_rank"]:
    l_ = CheckLog()
    try:
        sb.validate_submission(corrupt(kind), tpl_sub, metrics_wide, l_)
        caught = l_.failed
        which = ", ".join(l_.issues()["check_id"].tolist()[:3])
    except Exception as e:                       # lỗi cấu trúc nặng cũng tính là "bắt được"
        caught, which = True, f"dừng: {type(e).__name__}"
    neg.append({"kiểu lỗi": kind, "bị phát hiện": caught, "kiểm tra báo lỗi": which})
neg = pd.DataFrame(neg)
display(neg)
NEG_OK = bool(neg["bị phát hiện"].all())
print("Bộ kiểm tra bắt được mọi kiểu lỗi:", NEG_OK)

## 9. Manifest, lưu vết & báo cáo

In [ ]:
man = sb.manifest({"submission_output.csv": SUB_PATH, "qa_submission.csv": QA_PATH}, sha256_of)
man["run_id"], man["created_at"] = RUN_S7, now_iso()
man.to_csv(cfg.submit_dir / "submission_manifest.csv", index=False, encoding="utf-8-sig")
display(man)

# Bản lưu theo lần chạy (để so sánh các phiên bản nộp)
hist = cfg.submit_dir / "history" / RUN_S7
hist.mkdir(parents=True, exist_ok=True)
for p in (SUB_PATH, QA_PATH):
    shutil.copy2(p, hist / p.name)

save_output(vlog.df(), STEP, "validation_checks.csv")
save_output(layout, STEP, "pack_layout.csv")
save_output(neg, STEP, "negative_tests.csv")
append_audit({"run_id": RUN_S7, "step": "step7_submit", "parent_run": PARENTS, "verdict": VERDICT,
              "checks": vlog.counts(), "negative_tests_all_caught": NEG_OK,
              "pack_layout": {r.file: r.how for r in layout.itertuples()},
              "final_kpi_from_files": {k: round(float(v), 4) for k, v in final.items()},
              "pipeline_ms_total": round(ms_total, 1),
              "submission_sha256": dict(zip(man["file"], man["sha256"])),
              "outputs": ["05_submission/submission_output.csv", "05_submission/qa_submission.csv",
                          "05_submission/submission_manifest.csv", f"05_submission/history/{RUN_S7}/"]})

In [ ]:
pct = lambda x: f"{x:.1%}"
L = ["# Báo cáo NB3 – File nộp BTC", "",
     f"- **Run ID:** `{RUN_S7}` · **Thời điểm:** {now_iso()} · **Từ:** `{PARENTS[0]}` (NB1), `{PARENTS[1]}` (NB2)",
     f"- **Kiểm tra trước khi ghi:** **{VERDICT}** — {vlog.counts()}",
     f"- **Kiểm thử âm:** {'bắt được cả' if NEG_OK else 'KHÔNG bắt được hết'} {len(neg)} kiểu lỗi",
     f"- **Thời gian toàn pipeline:** {ms_total:.0f} ms / 60 company-year ({ms_total/60:.1f} ms/dòng); QA {qa_detail['processing_time_ms'].mean():.2f} ms/câu", "",
     "## File nộp", "", "| File | Dòng | Cột | SHA-256 |", "|---|---:|---:|---|"]
L += [f"| `{r.file}` | {r.rows} | {r.columns} | `{r.sha256[:16]}…` |" for r in man.itertuples()]
L += ["", "## KPI đo trực tiếp trên file nộp (đọc lại từ ổ đĩa)", "", "| KPI | Kết quả |", "|---|---:|"]
names = {"calculation_accuracy_file": "Calculation Accuracy (trong dung sai BTC)", "calculation_exact_file": "Chỉ tiêu khớp tuyệt đối",
         "numerical_qa_accuracy_file": "Numerical QA Accuracy", "alert_tag_f1_micro_file": "Cảnh báo – tag F1 (18 nhãn, in-sample)",
         "alert_scenario_accuracy_file": "Cảnh báo – kịch bản", "alert_severity_accuracy_file": "Cảnh báo – severity",
         "alert_health_accuracy_file": "Cảnh báo – health", "alert_high_critical_recall_file": "Recall HIGH/CRITICAL"}
L += [f"| {names[k]} | **{pct(v)}** |" for k, v in final.items() if k in names]
L += ["", "## Nhận diện Data Pack", "", "| File | Cách tìm thấy |", "|---|---|"]
L += [f"| {r.file} | {r.how} |" for r in layout.itertuples()]
L += ["", "## Phân bố cảnh báo trong file nộp", ""]
dist = pd.crosstab(submission["predicted_severity"], submission["predicted_overall_health"])
L += ["| Severity \\ Health | " + " | ".join(dist.columns) + " |", "|---|" + "---:|" * len(dist.columns)]
L += [f"| {i} | " + " | ".join(str(v) for v in r) + " |" for i, r in dist.iterrows()]
rep = cfg.step_dir(STEP) / "step7_report.md"
rep.write_text("\n".join(L), encoding="utf-8")
print("Đã ghi:", rep)
print("\nThư mục 05_submission:")
for p in sorted(cfg.submit_dir.iterdir()):
    print("  ", p.name)

## 10. Tải file nộp về máy (tùy chọn)

In [ ]:
try:
    from google.colab import files
    zip_path = shutil.make_archive(f"/content/submission_{RUN_S7}", "zip", cfg.submit_dir, ".")
    files.download(zip_path)
except ImportError:
    print("Không chạy trên Colab – file nằm tại", cfg.submit_dir)